# Exercise 20 - Big cities

Just as we can retrieve CSV-formatted data with ```pd.read_csv```, we can retrieve JSON-formatted data with ```pd.read_json```. In this exercise, I want you to read in data about the 1,000 largest cities in the United States. (This data is from 2013, so if your hometown doesn’t appear here, I apologize.) Once you have created a data frame from this city data, I want you to answer the following questions:

* What are the mean and median populations for these 1,000 largest cities? What does that tell you?
* Along these lines, if you remove the 50 most populous cities, what happens to the mean population? What happens to the median?
* What is the northernmost city, and where does it rank?
* Which state has the largest number of cities on this list?
* Which state has the smallest number of cities on this list?

In [24]:
import numpy as np
import pandas as pd

cities = pd.read_json('./cities.json')

cities.head(10)

,city,growth_from_2000_to_2013,latitude,longitude,population,rank,state
0,New York,4.8%,40.712784,-74.005941,8405837,1,New York
1,Los Angeles,4.8%,34.052234,-118.243685,3884307,2,California
2,Chicago,-6.1%,41.878114,-87.629798,2718782,3,Illinois
3,Houston,11.0%,29.760427,-95.369803,2195914,4,Texas
4,Philadelphia,2.6%,39.952584,-75.165222,1553165,5,Pennsylvania
5,Phoenix,14.0%,33.448377,-112.074037,1513367,6,Arizona
6,San Antonio,21.0%,29.424122,-98.493628,1409019,7,Texas
7,San Diego,10.5%,32.715738,-117.161084,1355896,8,California
8,Dallas,5.6%,32.776664,-96.796988,1257676,9,Texas
9,San Jose,10.5%,37.338208,-121.886329,998537,10,California


* What are the mean and median populations for these 1,000 largest cities? What does that tell you?

In [25]:
mean = cities["population"].mean()
median = cities["population"].median()

print(f'Mean of the population: {mean:,.2f}\nMedian of the population: {median:,.2f}')

Mean of the population: 131,132.44
Median of the population: 68,207.00


**Answer:** the mean and the median are very different which means that there are cities with very high population (outliers).

* Along these lines, if you remove the 50 most populous cities, what happens to the mean population? What happens to the median?

In [26]:
cities_2 = cities.sort_values(by="population", ascending=False).iloc[50:] # remove the 50 most populous cities

In [27]:
mean_2 = cities_2["population"].mean()
median_2 = cities_2["population"].median()

print(f"Mean of the population: {mean_2:,.2f}\nMedian of the population: {median_2:,.2f}")

Mean of the population: 87,027.39
Median of the population: 65,796.00


**Answer:** the mean and the median are much closer, which means that in the 50 most populous cities are many outliers.

* What is the northernmost city, and where does it rank?

In [28]:
# the northernmost city is the one with the max latitude
northernmost_city = cities.iloc[cities["latitude"].idxmax()]["city"]

# find where it ranks by population
sorted_cities = cities.sort_values(by="population", ascending=False)
city_rank = sorted_cities.query("city == @northernmost_city").index

print(f"The northernmost city is {northernmost_city} and is ranked {city_rank.tolist()[0]} in population")

The northernmost city is Anchorage and is ranked 62 in population


* Which state has the largest number of cities on this list?

In [29]:
state_most_cities = cities["state"].value_counts().head(1)
state_most_cities

state
California    212
Name: count, dtype: int64

* Which state has the smallest number of cities on this list?

In [30]:
state_less_cities = cities["state"].value_counts().tail(1)
state_less_cities

state
Vermont    1
Name: count, dtype: int64

## Beyond the exercise

* Convert the ```growth_from_2000_to_2013``` column into a floating-point number. Then find the mean and median changes in city size between 2000 and 2013. If a city has no recorded growth, set it to 0.

In [31]:
cities["growth_from_2000_to_2013"] = cities["growth_from_2000_to_2013"].str.rstrip("%") # remove % sign
cities.loc[cities["growth_from_2000_to_2013"] == "", "growth_from_2000_to_2013"] = "0" # cities with no growth have 0
cities["growth_from_2000_to_2013"] = cities["growth_from_2000_to_2013"].astype(np.float64) # convert to float64
cities["growth_from_2000_to_2013"].describe()[["mean", "50%"]] # find mean and median

mean    22.936
50%      9.650
Name: growth_from_2000_to_2013, dtype: float64

* How many cities had positive growth in this period, and how many had negative growth?

In [32]:
cities_pos_growth = cities.query("growth_from_2000_to_2013 > 0").shape[0]
cities_neg_growth = cities.query("growth_from_2000_to_2013 <= 0").shape[0] # if the city does not grow it is bad

print(f"Number of cities with positive growth: {cities_pos_growth}\nNumber of cities with negative growth: {cities_neg_growth}")

Number of cities with positive growth: 847
Number of cities with negative growth: 153


* Find the city or cities with latitudes more than two standard deviations from the mean.

In [33]:
mean_latitude = cities["latitude"].mean()
latitute_std = cities["latitude"].std()

cities.loc[
    (cities["latitude"] > mean_latitude + 2 * latitute_std) | \
    (cities["latitude"] < mean_latitude - 2 * latitute_std)
][["city", "latitude"]]

,city,latitude
43,Miami,25.761680
53,Honolulu,21.306944
62,Anchorage,61.218056
88,Hialeah,25.857596
130,Brownsville,25.901747
138,Fort Lauderdale,26.122439
145,Cape Coral,26.562854
149,Pembroke Pines,26.007765
173,Hollywood,26.011201
187,McAllen,26.203407
